# 02 - COVID-19 Data Cleaning

Here, we will analyze and correct dtypes, missing values, inconsistencies and duplicates.

## Import Libraries

In [1]:
import os
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
os.chdir(project_root)

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import the data loader module
from src.data_loader import load_csv, get_basic_info
from src.visualization import set_style

# Set visualization style
set_style()

print('Libraries imported successfully!')

Libraries imported successfully!


## Data Cleaning Strategy

- Detect and impute missing values
- Detect duplicated values
- Detect inconsistencies

## Load Raw Data

In [135]:
df = load_csv("data/raw/compact.csv")
df.head()

Data loaded successfully. Shape: (558807, 61)


,country,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,...,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand,human_development_index
0,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
1,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
2,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
3,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN
4,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,40578847.0,62.215549,16.752001,65.616997,1516.273315,NaN,10.9,51.938343,0.39,NaN


## Change dtypes

In [122]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

country                               object
date                          datetime64[ns]
total_cases                          float64
new_cases                            float64
new_cases_smoothed                   float64
                                   ...      
extreme_poverty                      float64
diabetes_prevalence                  float64
handwashing_facilities               float64
hospital_beds_per_thousand           float64
human_development_index              float64
Length: 61, dtype: object

### Filter until WHO said

In [123]:
df = df[df["date"] < "2023-05-01"]

## Missing Values

In [124]:
missing = (df.isna().mean()).sort_values(ascending=False)
missing

human_development_index                    1.000000
weekly_icu_admissions_per_million          0.970251
weekly_icu_admissions                      0.970251
excess_mortality_cumulative_per_million    0.962877
excess_mortality                           0.962769
                                             ...   
total_cases_per_million                    0.035078
total_deaths                               0.035078
total_deaths_per_million                   0.035078
date                                       0.000000
country                                    0.000000
Length: 61, dtype: float64

After looking at all missing values we get:

* Human Development Index: 100% missing -> drop
* Cases and death: low missing values
* Excess mortality: +97% -> drop
* ICU and Hospital patients: ~93% -> use with available (can be interesting)
* Stringency index: 63% -> use with available data
* Reproduction rate: 66% -> use with available
* Tests: ~85% -> use with available (robust testing such as EU, USA, China, etc.)
* Vaccination: ~85% -> use only with available
* Handwashing, hospital beds, extreme poverty, gdp: useful for analysis and with a low rate of missing (~ <50%)

**Practical Strategy**
1. Mandatory variables: cases, deaths, country, continent, population
2. Optional variables (for sub-analysis with complete data): vaccination, stringency and reproduction rate, hospital and icu, tests
3. Context or socioeconomical variables: gdp, median age, life expectancy, hospital beds, handwashing.
4. Drop: excess mortality, weekly ICU/hospital admissions, total boosters

**Imputation tips and cleaning**

* Socioeconomic variables: impute with the median by continent.
* Vaccination variables and tests: not impute with 0, use only countries with data in each analysis
* Rolling averages (new_cases_smoothed): use them for temporal plots.

### Dropping columns

In [125]:
cols_to_drop = [
    "human_development_index",
    "weekly_icu_admissions",
    "weekly_icu_admissions_per_million",
    "excess_mortality",
    "excess_mortality_cumulative",
    "excess_mortality_cumulative_absolute",
    "excess_mortality_cumulative_per_million",
    "total_boosters",
    "total_boosters_per_hundred"
]

df = df.drop(columns=cols_to_drop, axis=1)

### Optional Variables

In [126]:
vacc_df = df.dropna(subset=["people_vaccinated"]).copy()

### Socioeconomic Variables

In [127]:
for col in ["gdp_per_capita","median_age","life_expectancy","hospital_beds_per_thousand","handwashing_facilities","extreme_poverty"]:
    df[col] = df.groupby("continent")[col].transform(lambda x: x.fillna(x.median()))

## Duplicated values

In [128]:
duplicates = df.duplicated(subset=["country", "date"]).isna().sum()
print(f"Number of duplicated values: {duplicates}")

Number of duplicated values: 0


We can follow the analysis.

## Inconsistencies

### Numerical variables

In [129]:
df.describe()

,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,new_deaths_smoothed,...,new_people_vaccinated_smoothed_per_hundred,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand
count,313017,3.020370e+05,3.020070e+05,3.007780e+05,302037.000000,302007.000000,300778.000000,3.020370e+05,302015.000000,300792.000000,...,170022.000000,3.012940e+05,296439.000000,287991.000000,287991.000000,287991.000000,287991.000000,262468.000000,287991.000000,287991.000000
mean,2021-08-31 23:45:17.553359616,8.735293e+06,1.766032e+04,1.772351e+04,78138.713066,163.609281,164.235402,1.168204e+05,168.089764,168.701167,...,0.081806,1.310441e+08,419.608631,31.195932,73.453035,21748.714632,11.324572,9.044009,73.127703,2.714319
min,2020-01-01 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,...,0.000000,5.130000e+02,0.136470,14.298000,18.817699,708.178284,0.000000,1.100000,3.487560,0.300000
25%,2020-11-02 00:00:00,2.352000e+03,0.000000e+00,1.142857e+00,843.017640,0.000000,0.271394,2.100000e+01,0.000000,0.000000,...,0.002297,5.197500e+05,36.265862,22.243999,68.748703,5579.981445,0.654043,5.600000,58.306389,1.000000
50%,2021-09-02 00:00:00,3.914400e+04,8.000000e+00,4.157143e+01,12886.612000,1.038304,11.945877,5.140000e+02,0.000000,0.285714,...,0.018079,6.280316e+06,92.084663,31.683001,74.695999,14918.296875,2.436315,7.450000,87.331657,2.040000
75%,2022-07-01 00:00:00,5.506240e+05,4.680000e+02,7.395714e+02,91956.550000,52.913403,105.863162,8.064000e+03,5.000000,7.857143,...,0.085231,2.971544e+07,237.470367,39.080002,79.044800,34297.933594,10.221504,11.100000,97.261055,3.490000
max,2023-04-30 00:00:00,7.643216e+08,8.401906e+06,6.402033e+06,707442.940000,230762.550000,37463.746000,6.927005e+06,57167.000000,14820.714000,...,11.734285,8.021407e+09,21344.242188,59.875000,85.746399,117746.992188,85.317673,30.799999,100.000000,13.800000
std,NaN,5.217395e+07,1.528092e+05,1.305465e+05,131720.497618,1263.294651,648.124492,6.308909e+05,1044.503861,963.278472,...,0.184960,6.671862e+08,1931.030592,9.714021,7.720065,21297.374933,18.507904,5.234072,28.281346,2.231152


We can see that there are not impossible values, so these variables are good for the moment.

### Categorical variables

In [130]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical variables:", categorical_cols)

Categorical variables: ['country', 'code', 'continent']


In [131]:
for col in categorical_cols:
    df[col] = df[col].astype("category")

We transformed the categorical columns into `category` dtype for pandas optimization.

## Save Processed Data

In [132]:
df.to_csv("data/processed/df_master.csv", index=False)

In [133]:
df_master = pd.read_csv("data/processed/df_master.csv", parse_dates=["date"])
df_master.head()

,country,date,total_cases,new_cases,new_cases_smoothed,total_cases_per_million,new_cases_per_million,new_cases_smoothed_per_million,total_deaths,new_deaths,...,continent,population,population_density,median_age,life_expectancy,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities,hospital_beds_per_thousand
0,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Asia,40578847.0,62.215549,16.752001,65.616997,1516.273315,2.436315,10.9,51.938343,0.39
1,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Asia,40578847.0,62.215549,16.752001,65.616997,1516.273315,2.436315,10.9,51.938343,0.39
2,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Asia,40578847.0,62.215549,16.752001,65.616997,1516.273315,2.436315,10.9,51.938343,0.39
3,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,Asia,40578847.0,62.215549,16.752001,65.616997,1516.273315,2.436315,10.9,51.938343,0.39
4,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,...,Asia,40578847.0,62.215549,16.752001,65.616997,1516.273315,2.436315,10.9,51.938343,0.39


Now, as we had analyzed before, we want to divide the data into country and continent.

In order to resolve this problem, we will save a dataset of countries and another of not countries (continents, group of countries, etc.) in the `data/preprocessed` folder.

In [134]:
countries = df_master[df_master["continent"].notna()].copy()
countries.to_csv("data/processed/countries.csv", index=False)

not_countries = df_master[df_master["continent"].isna()].copy()
not_countries.to_csv("data/processed/not_countries.csv", index=False)

Once we have this done, we can work separately and easier with each dataset. We should take into account the `dtypes` already mentioned and the variable selection for each question. We will do this in the next code.

However, before performing any analysis we should clean our data first, in order to see missing values, inconsistencies and duplicates.